In [30]:
# Install wandb for experiment tracking, transformers for AST, timm for ViT.
# Run this cell FIRST — Kaggle kernels may not have all of these by default.

import subprocess
subprocess.run(["pip", "install", "wandb", "transformers", "timm", "scikit-image", "-q"])

CompletedProcess(args=['pip', 'install', 'wandb', 'transformers', 'timm', 'scikit-image', '-q'], returncode=0)

In [54]:
# All library imports in one place. AMP (automatic mixed precision) is used
# for faster GPU training. HuggingFace is optional — we fall back to timm.

import os
import gc
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")

import librosa
import librosa.display
from skimage.transform import resize as sk_resize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

import timm
try:
    from transformers import ASTForAudioClassification
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    print("HuggingFace not available — will use timm ViT as AST substitute")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report

import wandb

def set_seed(seed=42):
    """Fix all random seeds for reproducibility across runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Device: cuda
GPU : Tesla T4
VRAM: 15.6 GB


In [55]:
# Kaggle competition datasets can be mounted at different slugs depending on
# how the dataset was added. This cell walks /kaggle/input and finds all the
# required folders automatically — no manual path editing needed.

def find_dir(root: str, target_name: str) -> str:
    """
    Recursively search under `root` for a directory whose name == target_name.
    Returns the first match as a string path, or raises FileNotFoundError.
    """
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d == target_name:
                return os.path.join(dirpath, d)
    raise FileNotFoundError(
        f"Could not find directory '{target_name}' under '{root}'. "
        f"Check that your dataset is attached to this notebook."
    )

def find_file(root: str, filename: str) -> str:
    """
    Recursively search under `root` for a file named `filename`.
    Returns the first match as a string path, or raises FileNotFoundError.
    """
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    raise FileNotFoundError(
        f"Could not find file '{filename}' under '{root}'."
    )

KAGGLE_INPUT = "/kaggle/input"
print("Scanning /kaggle/input for dataset directories…")
print("Mounted datasets:", os.listdir(KAGGLE_INPUT))

# Auto-locate each required path
GENRES_STEMS_DIR = find_dir(KAGGLE_INPUT, "genres_stems")
MASHUPS_DIR      = find_dir(KAGGLE_INPUT, "mashups")
ESC50_AUDIO_DIR  = find_dir(KAGGLE_INPUT, "audio")  # ESC-50-master/audio
TEST_CSV         = find_file(KAGGLE_INPUT, "test.csv")
SAMPLE_SUB_CSV   = find_file(KAGGLE_INPUT, "sample_submission.csv")

print(f"\nPaths resolved:")
print(f"  genres_stems : {GENRES_STEMS_DIR}")
print(f"  mashups      : {MASHUPS_DIR}")
print(f"  ESC-50 audio : {ESC50_AUDIO_DIR}")
print(f"  test.csv     : {TEST_CSV}")
print(f"  sample_sub   : {SAMPLE_SUB_CSV}")

# Quick sanity check
for d in [GENRES_STEMS_DIR, MASHUPS_DIR, ESC50_AUDIO_DIR]:
    n = len(list(Path(d).iterdir()))
    print(f"  [{n} items] {d}")

Scanning /kaggle/input for dataset directories…
Mounted datasets: ['jan-2026-dl-gen-ai-project']

Paths resolved:
  genres_stems : /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems
  mashups      : /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups
  ESC-50 audio : /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio
  test.csv     : /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv
  sample_sub   : /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv
  [10 items] /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems
  [3020 items] /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups
  [2000 items] /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio


In [56]:
# All hyperparameters in one dict. Change values here only.

CONFIG = {
    # ---- Dataset paths (filled from auto-detection above) ----
    "genres_stems_dir": GENRES_STEMS_DIR,
    "mashups_dir":      MASHUPS_DIR,
    "esc50_audio_dir":  ESC50_AUDIO_DIR,
    "test_csv":         TEST_CSV,
    "sample_sub_csv":   SAMPLE_SUB_CSV,

    # ---- Audio parameters ----
    "sample_rate":  22050,  # Hz — standard for music
    "duration":     20,     # seconds to load per clip
    "n_mels":       128,    # mel filterbank channels
    "n_fft":        2048,   # FFT window size
    "hop_length":   512,    # FFT hop
    "fmax":         8000,   # max mel frequency

    # ---- Spectrogram image size ----
    "img_height":   128,    # = n_mels
    "img_width":    256,    # time frames (resized to this)

    # ---- Training hyperparameters ----
    "batch_size":         64,
    "num_workers":        2,
    "val_split":          0.15,
    "num_epochs_cnn":     5,
    "num_epochs_resnet":  3,
    "num_epochs_ast":     2,
    "lr_cnn":             1e-3,
    "lr_resnet":          5e-4,
    "lr_ast":             1e-4,
    "weight_decay":       1e-4,
    "label_smoothing":    0.1,
    "grad_clip":          1.0,

    # ---- Augmentation ----
    "aug_per_song": 1,   # extra mashup-style samples per training song
    "num_tta":      5,   # test-time augmentation rounds

    # ---- Classes ----
    "num_classes": 10,
    "genres": [
        "blues", "classical", "country", "disco", "hiphop",
        "jazz", "metal", "pop", "reggae", "rock"
    ],

    # ---- WandB ----
     "wandb_project": "23f3004501-t12026",
    "wandb_entity":  "23f3004501-indian-institute-of-technology-madras",
}

print("Configuration loaded. Genres:", CONFIG["genres"])

Configuration loaded. Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


In [57]:
# Paste your WandB API key below. You can also store it in Kaggle Secrets
# (Add-ons → Secrets → add key "WANDB_API_KEY") and read it with:
#   os.environ["WANDB_API_KEY"] = os.environ.get("WANDB_API_KEY", "")

WANDB_API_KEY = "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"   # <-- REPLACE WITH YOUR ACTUAL KEY

# If you used Kaggle Secrets, uncomment the next line instead:
# from kaggle_secrets import UserSecretsClient
# WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY

def init_wandb_run(run_name: str, extra_cfg: dict = None):
    """
    Start a new WandB run. Each model (CNN / ResNet / AST) gets its own run
    so you can compare learning curves side-by-side on the WandB dashboard.
    """
    cfg = {k: v for k, v in CONFIG.items() if not isinstance(v, (list, dict))}
    if extra_cfg:
        cfg.update(extra_cfg)
    run = wandb.init(
        project=CONFIG["wandb_project"],
        entity=CONFIG["wandb_entity"],
        name=run_name,
        config=cfg,
        reinit=True,
    )
    print(f"WandB run started: {run.name}  |  {run.url}")
    return run

print("WandB helper ready!")

WandB helper ready!


In [59]:
def load_audio(path: str,
               sr=CONFIG["sample_rate"],
               duration=CONFIG["duration"]) -> np.ndarray:
    """
    Load a WAV file as a mono float32 array.
    - Pads with zeros if shorter than `duration` seconds.
    - Truncates if longer.
    Returns array of shape (sr * duration,).
    """
    target_len = sr * duration
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)), mode="constant")
        else:
            y = y[:target_len]
        return y.astype(np.float32)
    except Exception as e:
        print(f"  [WARN] Could not load {path}: {e}")
        return np.zeros(target_len, dtype=np.float32)


def mel_spectrogram(y: np.ndarray,
                    sr=CONFIG["sample_rate"],
                    n_mels=CONFIG["n_mels"],
                    n_fft=CONFIG["n_fft"],
                    hop_length=CONFIG["hop_length"],
                    fmax=CONFIG["fmax"]) -> np.ndarray:
    """
    Compute a log-mel spectrogram from waveform y.
    Steps: power mel spec → convert to dB (log compression).
    Returns array of shape (n_mels, time_frames).
    """
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels,
        n_fft=n_fft, hop_length=hop_length, fmax=fmax
    )
    return librosa.power_to_db(S, ref=np.max).astype(np.float32)


def normalize_spec(spec: np.ndarray) -> np.ndarray:
    """Min-max normalize spectrogram to [0, 1]."""
    lo, hi = spec.min(), spec.max()
    return (spec - lo) / (hi - lo + 1e-6)


def resize_spec(spec: np.ndarray,
                h=CONFIG["img_height"],
                w=CONFIG["img_width"]) -> np.ndarray:
    """
    Resize spectrogram to fixed (H, W) so all model inputs are identical.
    Uses anti-aliased bicubic interpolation.
    """
    return sk_resize(spec, (h, w), anti_aliasing=True).astype(np.float32)


def wav_to_spec(path: str) -> np.ndarray:
    """Full pipeline: WAV path → normalized, resized mel spectrogram (H, W)."""
    y    = load_audio(path)
    spec = mel_spectrogram(y)
    spec = normalize_spec(spec)
    return resize_spec(spec)


print("Audio utility functions ready!")

Audio utility functions ready!


In [61]:
# KEY CHALLENGE: training distribution (clean stems) ≠ test distribution
# (noisy cross-song mashups). We bridge this gap with three augmentations:
#   1. Cross-song stem mixing — mix stems from different songs, same genre
#   2. Tempo perturbation     — simulate BPM alignment
#   3. ESC-50 noise injection — simulate real-world noise

def load_esc50_files(esc50_audio_dir: str) -> list:
    """
    Collect all WAV files from the ESC-50 noise directory.
    Tries both .wav and .WAV extensions and subdirectories.
    Returns list of Path objects (may be empty if dir is missing).
    """
    p = Path(esc50_audio_dir)
    if not p.exists():
        print(f"  [WARN] ESC-50 audio dir not found: {esc50_audio_dir}")
        return []
    files = [
    Path(e.path)
    for e in os.scandir(str(p))
    if e.is_file() and e.name.lower().endswith(".wav")
]
    print(f"Found {len(files)} ESC-50 noise files in {esc50_audio_dir}")
    return files


def mix_cross_song_stems(genre_dir: str) -> np.ndarray:
    """
    Synthesise a mashup: each of the 4 stems (drums, vocals, bass, others)
    is taken from a DIFFERENT randomly chosen song of the same genre.
    This directly mirrors how the test set was constructed.

    - Random per-stem volume scaling simulates instrument balance variation.
    - Peak-normalises the result to prevent clipping.
    Returns mixed mono waveform of length (sr * duration,).
    """
    sr, dur    = CONFIG["sample_rate"], CONFIG["duration"]
    target_len = sr * dur
    song_dirs  = [d for d in Path(genre_dir).iterdir() if d.is_dir()]

    # Safety: if fewer than 2 songs just repeat
    if len(song_dirs) < 2:
        song_dirs = song_dirs * 4

    stem_names = ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]
    mixed      = np.zeros(target_len, dtype=np.float32)

    for stem_name in stem_names:
        song_dir  = random.choice(song_dirs)
        stem_path = song_dir / stem_name

        # Handle "other.wav" vs "others.wav" naming difference
        if not stem_path.exists():
            alt = song_dir / "other.wav"
            if alt.exists():
                stem_path = alt

        if stem_path.exists():
            y      = load_audio(str(stem_path))
            scale  = random.uniform(0.4, 1.6)   # random instrument balance
            mixed += y * scale

    peak = np.max(np.abs(mixed))
    if peak > 1e-6:
        mixed = mixed / peak * 0.9
    return mixed


def add_esc50_noise(y: np.ndarray,
                    noise_files: list,
                    snr_range=(5, 25)) -> np.ndarray:
    """
    Additively mix a random ESC-50 clip into `y` at a random SNR (dB).
    - SNR  5 dB → heavy noise  |  SNR 25 dB → light noise
    - 10% chance of skipping noise entirely.
    - If no noise files are available, returns y unchanged.
    Returns: noisy waveform, same shape as y.
    """
    if not noise_files or random.random() < 0.10:
        return y

    noise_path = random.choice(noise_files)
    noise      = load_audio(str(noise_path))

    snr_db  = random.uniform(*snr_range)
    sig_pwr = np.mean(y ** 2) + 1e-10
    nse_pwr = np.mean(noise ** 2) + 1e-10
    scale   = np.sqrt(sig_pwr / ((10 ** (snr_db / 10)) * nse_pwr))

    noisy = y + noise * scale
    peak  = np.max(np.abs(noisy))
    if peak > 1e-6:
        noisy = noisy / peak * 0.9
    return noisy.astype(np.float32)


def tempo_perturb(y: np.ndarray, rate_range=(0.85, 1.15)) -> np.ndarray:
    """
    Time-stretch (or compress) the waveform without changing pitch.
    Simulates the BPM alignment applied to stems during mashup creation.
    Output is trimmed / padded back to the original length.
    """
    rate  = random.uniform(*rate_range)
    y_out = librosa.effects.time_stretch(y, rate=rate)
    tgt   = CONFIG["sample_rate"] * CONFIG["duration"]
    if len(y_out) < tgt:
        y_out = np.pad(y_out, (0, tgt - len(y_out)))
    return y_out[:tgt].astype(np.float32)


def spec_augment(spec: np.ndarray,
                 freq_param=20,
                 time_param=30,
                 n_masks=2) -> np.ndarray:
    """
    SpecAugment (Park et al. 2019): zero out random frequency and time bands.
    Applied ON-LINE during training to reduce overfitting.
    - freq_param: max mel-bin rows to mask per mask
    - time_param: max time-frame columns to mask per mask
    - n_masks:    how many masks to apply in each dimension
    Returns: augmented copy (original is not modified).
    """
    spec = spec.copy()
    H, W = spec.shape

    for _ in range(n_masks):
        f  = random.randint(0, freq_param)
        f0 = random.randint(0, max(0, H - f))
        spec[f0:f0 + f, :] = 0.0

        t  = random.randint(0, time_param)
        t0 = random.randint(0, max(0, W - t))
        spec[:, t0:t0 + t] = 0.0

    return spec


print("Augmentation functions ready!")

Augmentation functions ready!


In [62]:
# ===========================================================================
# CELL 8 — Build Training Dataset (Parallel)
# ===========================================================================
# Feature extraction runs on CPU regardless of GPU availability because
# librosa/numpy/skimage have no GPU backend. The speedup comes from
# parallelising across all CPU cores with ProcessPoolExecutor so multiple
# songs are processed simultaneously instead of one at a time.
#
# Flow:
#   1. Build a flat list of tasks — one task per song (all genres)
#   2. Dispatch all tasks to N_WORKERS parallel processes
#   3. Each worker: original mix + aug_per_song augmented mashups → specs
#   4. Collect results via as_completed() with a live tqdm progress bar

import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

# Kaggle gives 4 CPU cores — use all but 1 to keep the main process responsive
N_WORKERS = max(1, multiprocessing.cpu_count() - 1)
print(f"CPU cores available: {multiprocessing.cpu_count()}  |  Workers: {N_WORKERS}")


def _process_one_song(args):
    """
    Worker function — runs inside a subprocess (must be a top-level function
    so Python can pickle it). Processes ONE song directory and returns a list
    of (spec_array, label) tuples: 1 original + aug_per_song augmented.

    All libraries are imported locally because each worker process starts
    fresh and does not inherit the parent's global namespace.
    All paths are passed as plain strings (Path objects aren't always safely
    picklable across process boundaries).
    """
    import random
    import numpy as np
    import librosa
    from pathlib import Path
    from skimage.transform import resize as sk_resize

    (song_dir_str, genre_path_str, label,
     noise_file_strs, aug_per_song,
     SR, DUR, N_MELS, N_FFT, HOP, FMAX, H, W) = args

    song_dir   = Path(song_dir_str)
    genre_path = Path(genre_path_str)
    target_len = SR * DUR

    # ---- inline utility functions (no globals available in worker) ----

    def _load(path):
        """Load WAV → fixed-length mono float32 array."""
        try:
            y, _ = librosa.load(path, sr=SR, duration=DUR, mono=True)
            if len(y) < target_len:
                y = np.pad(y, (0, target_len - len(y)))
            return y[:target_len].astype(np.float32)
        except Exception:
            return np.zeros(target_len, dtype=np.float32)

    def _to_spec(y):
        """Waveform → normalised, resized log-mel spectrogram."""
        S  = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS,
            n_fft=N_FFT, hop_length=HOP, fmax=FMAX)
        db = librosa.power_to_db(S, ref=np.max).astype(np.float32)
        lo, hi = db.min(), db.max()
        db = (db - lo) / (hi - lo + 1e-6)                 # normalise [0,1]
        return sk_resize(db, (H, W), anti_aliasing=True).astype(np.float32)

    def _mix_cross(all_song_dirs):
        """Mix stems from DIFFERENT songs of the same genre (cross-song mashup)."""
        mixed = np.zeros(target_len, dtype=np.float32)
        for stem in ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]:
            sd = random.choice(all_song_dirs)
            sp = sd / stem
            if not sp.exists():
                sp = sd / "other.wav"   # handle naming variant
            if sp.exists():
                mixed += _load(str(sp)) * random.uniform(0.4, 1.6)
        peak = np.max(np.abs(mixed))
        return mixed / peak * 0.9 if peak > 1e-6 else mixed

    def _add_noise(y, noise_files):
        """Add ESC-50 noise at a random SNR between 5–25 dB (10% chance skip)."""
        if not noise_files or random.random() < 0.10:
            return y
        noise  = _load(random.choice(noise_files))
        snr    = random.uniform(5, 25)
        sp     = np.mean(y ** 2) + 1e-10
        np_    = np.mean(noise ** 2) + 1e-10
        scale  = np.sqrt(sp / ((10 ** (snr / 10)) * np_))
        noisy  = y + noise * scale
        peak   = np.max(np.abs(noisy))
        return (noisy / peak * 0.9).astype(np.float32) if peak > 1e-6 else noisy

    def _tempo(y):
        """Time-stretch waveform by a random rate 0.85–1.15× (pitch preserved)."""
        rate  = random.uniform(0.85, 1.15)
        y_out = librosa.effects.time_stretch(y, rate=rate)
        if len(y_out) < target_len:
            y_out = np.pad(y_out, (0, target_len - len(y_out)))
        return y_out[:target_len].astype(np.float32)

    # ------------------------------------------------------------------ #
    results = []

    # 1. Original: sum all 4 stems of THIS song
    mixed, found = np.zeros(target_len, dtype=np.float32), 0
    for stem in ["drums.wav", "vocals.wav", "bass.wav", "others.wav", "other.wav"]:
        sp = song_dir / stem
        if sp.exists():
            mixed += _load(str(sp))
            found += 1

    if found == 0:
        return results  # skip songs with no readable stems

    peak = np.max(np.abs(mixed))
    if peak > 1e-6:
        mixed = mixed / peak * 0.9
    results.append((_to_spec(mixed), label))

    # 2. Augmented cross-song mashups
    all_song_dirs = [d for d in genre_path.iterdir() if d.is_dir()]
    if len(all_song_dirs) < 2:
        all_song_dirs = all_song_dirs * 4  # safety fallback for tiny genres

    for _ in range(aug_per_song):
        aug = _mix_cross(all_song_dirs)
        if random.random() > 0.5:
            aug = _tempo(aug)           # 50% chance: tempo perturbation
        aug = _add_noise(aug, noise_file_strs)
        results.append((_to_spec(aug), label))

    return results


def build_training_dataset(genres_stems_dir, esc50_audio_dir,
                           aug_per_song=CONFIG["aug_per_song"]):
    """
    Parallel feature extraction — dispatches one task per song to
    N_WORKERS subprocesses and collects results as they finish.

    Returns:
        samples       : list of (np.ndarray H×W, int label)
        label_encoder : fitted sklearn LabelEncoder
    """
    genres = CONFIG["genres"]
    le     = LabelEncoder().fit(genres)

    # Noise file paths as plain strings — safe to pickle across processes
    noise_file_strs = [str(f) for f in load_esc50_files(esc50_audio_dir)]

    # Audio params tuple passed explicitly to each worker
    audio_params = (
        CONFIG["sample_rate"], CONFIG["duration"],
        CONFIG["n_mels"], CONFIG["n_fft"],
        CONFIG["hop_length"], CONFIG["fmax"],
        CONFIG["img_height"], CONFIG["img_width"],
    )

    # Build flat task list — one entry per song across all genres
    tasks = []
    for genre in genres:
        genre_path = Path(genres_stems_dir) / genre
        if not genre_path.exists():
            print(f"  [WARN] Missing genre folder: {genre_path}")
            continue
        label     = int(le.transform([genre])[0])
        song_dirs = sorted([d for d in genre_path.iterdir() if d.is_dir()])
        if not song_dirs:
            print(f"  [WARN] No song dirs in {genre_path}")
            continue
        for song_dir in song_dirs:
            tasks.append((
                str(song_dir), str(genre_path), label,
                noise_file_strs, aug_per_song,
                *audio_params
            ))

    print(f"\nDispatching {len(tasks)} songs → {N_WORKERS} parallel workers…")
    print(f"Expected: ~{len(tasks) * (1 + aug_per_song)} total spectrograms")

    samples = []
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(_process_one_song, t): t for t in tasks}
        with tqdm(total=len(tasks), desc="Extracting spectrograms") as pbar:
            for future in as_completed(futures):
                try:
                    samples.extend(future.result())
                except Exception as e:
                    print(f"  [Worker error] {e}")
                pbar.update(1)

    print(f"\nTotal training samples: {len(samples)}")
    if not samples:
        raise RuntimeError("No samples extracted — check genres_stems_dir path.")

    genre_map = {i: g for i, g in enumerate(genres)}
    dist = pd.Series([s[1] for s in samples]).map(genre_map).value_counts()
    print("Label distribution:\n", dist.to_string())
    return samples, le


# ---- RUN ----
print("Starting parallel feature extraction…")
all_samples, label_encoder = build_training_dataset(
    CONFIG["genres_stems_dir"],
    CONFIG["esc50_audio_dir"],
    aug_per_song=CONFIG["aug_per_song"],
)
print("Feature extraction complete!")

CPU cores available: 4  |  Workers: 3
Starting parallel feature extraction…
Found 2000 ESC-50 noise files in /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio

Dispatching 1000 songs → 3 parallel workers…
Expected: ~2000 total spectrograms


Extracting spectrograms: 100%|██████████| 1000/1000 [04:53<00:00,  3.41it/s]



Total training samples: 2000
Label distribution:
 blues        200
classical    200
country      200
disco        200
hiphop       200
jazz         200
metal        200
pop          200
reggae       200
rock         200
Feature extraction complete!


In [63]:
# The test.csv file maps IDs to mashup audio filenames.
# We try multiple filename formats since Kaggle datasets sometimes differ.

print("\nReading test.csv…")
test_df = pd.read_csv(CONFIG["test_csv"])
print(f"Test entries: {len(test_df)}")
print(test_df.head())
print("Columns:", list(test_df.columns))

# Peek at what files are actually in the mashups directory
mashup_files = sorted(Path(CONFIG["mashups_dir"]).glob("*.wav"))
print(f"\nMashup directory contains {len(mashup_files)} WAV files")
print("First 5:", [f.name for f in mashup_files[:5]])

def resolve_test_path(row, mashups_dir: str) -> str:
    """
    Try multiple filename patterns to locate each test audio file.
    Handles: song0001.wav / song_0001.wav / 0001.wav / numeric id directly.
    """
    mid = str(row["id"])
    candidates = [
        f"song{mid.zfill(4)}.wav",
        f"song_{mid.zfill(4)}.wav",
        f"{mid.zfill(4)}.wav",
        f"{mid}.wav",
    ]
    # Also check if a 'filename' or 'audio' column exists in test.csv
    if "filename" in row:
        candidates.insert(0, row["filename"])
    if "audio" in row:
        candidates.insert(0, row["audio"])

    for fn in candidates:
        p = os.path.join(mashups_dir, fn)
        if os.path.exists(p):
            return p

    # Last resort: just return the most likely path (will be zeroed on load)
    return os.path.join(mashups_dir, candidates[0])


test_paths = [resolve_test_path(row, CONFIG["mashups_dir"])
              for _, row in test_df.iterrows()]

found = sum(1 for p in test_paths if os.path.exists(p))
print(f"\nTest files resolved: {found}/{len(test_paths)} found on disk")
if found == 0:
    # Fallback: list actual mashup files and align by index
    print("  [FALLBACK] Aligning test by sorted file list…")
    mashup_names = sorted([f.name for f in mashup_files])
    test_paths = [os.path.join(CONFIG["mashups_dir"], mashup_names[i])
                  if i < len(mashup_names) else test_paths[i]
                  for i in range(len(test_df))]
    found = sum(1 for p in test_paths if os.path.exists(p))
    print(f"  After fallback: {found}/{len(test_paths)} found")


Reading test.csv…
Test entries: 3020
   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav
Columns: ['id', 'filename']

Mashup directory contains 3020 WAV files
First 5: ['song0001.wav', 'song0002.wav', 'song0003.wav', 'song0004.wav', 'song0005.wav']

Test files resolved: 3020/3020 found on disk


In [64]:
class TrainDataset(Dataset):
    """
    Wraps pre-computed (spec, label) pairs.
    Applies SpecAugment online during training to further regularize.
    Stacks the single-channel spec to 3 channels for ImageNet-style models.
    """

    def __init__(self, samples: list, is_train: bool = True):
        self.samples  = samples
        self.is_train = is_train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        spec, label = self.samples[idx]

        # Online SpecAugment — only during training
        if self.is_train and random.random() > 0.3:
            spec = spec_augment(spec, freq_param=20, time_param=30, n_masks=2)

        # (H, W) → (3, H, W) by repeating the grey channel 3 times
        tensor = torch.from_numpy(spec).unsqueeze(0).repeat(3, 1, 1)
        return tensor, torch.tensor(label, dtype=torch.long)


class TestDataset(Dataset):
    """
    Pre-computes mel spectrograms for all test mashup files.
    Stores everything in memory for fast repeated TTA inference.
    """

    def __init__(self, file_paths: list):
        self.specs = []
        print("Pre-computing test spectrograms…")
        for fp in tqdm(file_paths, desc="Test audio"):
            self.specs.append(wav_to_spec(fp))
        print(f"Loaded {len(self.specs)} test spectrograms.")

    def __len__(self):
        return len(self.specs)

    def __getitem__(self, idx):
        return torch.from_numpy(self.specs[idx]).unsqueeze(0).repeat(3, 1, 1)


# ---- Stratified train / val split ----
print("\nSplitting into train / val…")
train_samp, val_samp = train_test_split(
    all_samples,
    test_size=CONFIG["val_split"],
    stratify=[s[1] for s in all_samples],
    random_state=42,
)
print(f"Train: {len(train_samp)}  |  Val: {len(val_samp)}")

# ---- DataLoaders ----
NW = CONFIG["num_workers"]

train_dl = DataLoader(
    TrainDataset(train_samp, is_train=True),
    batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=NW, pin_memory=True, drop_last=True,
)
val_dl = DataLoader(
    TrainDataset(val_samp, is_train=False),
    batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=NW, pin_memory=True,
)

print(f"Train batches: {len(train_dl)}  |  Val batches: {len(val_dl)}")

# ---- Test DataLoader ----
test_ds = TestDataset(test_paths)
test_dl = DataLoader(
    test_ds, batch_size=CONFIG["batch_size"],
    shuffle=False, num_workers=NW, pin_memory=True,
)
print("All data loaders ready!")


Splitting into train / val…
Train: 1700  |  Val: 300
Train batches: 26  |  Val batches: 5
Pre-computing test spectrograms…


Test audio: 100%|██████████| 3020/3020 [02:33<00:00, 19.66it/s]

Loaded 3020 test spectrograms.
All data loaders ready!


In [65]:
# 5-block CNN designed specifically for mel spectrograms.
# No pretrained weights — learns entirely from our augmented data.

class ConvBlock(nn.Module):
    """Conv2d → BatchNorm2d → ReLU (+ optional spatial Dropout)"""

    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, drop=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(drop) if drop > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class ScratchCNN(nn.Module):
    """
    5-block CNN for music genre classification.
    Architecture:
        Block 1: 3  → 32  ch  | 128×256 → 64×128
        Block 2: 32 → 64  ch  |  64×128 → 32×64
        Block 3: 64 → 128 ch  |  32×64  → 16×32
        Block 4: 128→ 256 ch  |  16×32  →  8×16
        Block 5: 256→ 512 ch  |   8×16  →  4×8
        GAP                   →  512-d vector
        MLP: 512 → 256 → 128 → num_classes

    Design decisions:
    - Increasing channel width captures richer frequency/time patterns.
    - Global Average Pooling avoids overfitting compared to large FC layers.
    - Dropout2d + Dropout regularize at both spatial and neuron levels.
    """

    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32,  drop=0.10), ConvBlock(32,  32),  nn.MaxPool2d(2, 2),
            ConvBlock(32,  64,  drop=0.10), ConvBlock(64,  64),  nn.MaxPool2d(2, 2),
            ConvBlock(64,  128, drop=0.15), ConvBlock(128, 128), nn.MaxPool2d(2, 2),
            ConvBlock(128, 256, drop=0.20), ConvBlock(256, 256), nn.MaxPool2d(2, 2),
            ConvBlock(256, 512, drop=0.25), ConvBlock(512, 512), nn.MaxPool2d(2, 2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


# Sanity check
_m = ScratchCNN(10).to(DEVICE)
_d = torch.randn(2, 3, 128, 256).to(DEVICE)
print("CNN output shape:", _m(_d).shape)
print(f"CNN params: {sum(p.numel() for p in _m.parameters()):,}")
del _m, _d; torch.cuda.empty_cache()

CNN output shape: torch.Size([2, 10])
CNN params: 4,879,722


In [66]:
# Pretrained on ImageNet — repurposed for mel spectrogram classification.
# Two-stage fine-tuning avoids catastrophic forgetting on the small dataset.

class ResNet50Genre(nn.Module):
    """
    ResNet-50 fine-tuned for genre classification.

    Transfer learning strategy:
      Stage 1 (epochs 1–10):
          Freeze stem, layer1, layer2. Only train layer3, layer4, and FC head.
          This quickly adapts the deep features without corrupting early
          low-level ImageNet representations.
      Stage 2 (epoch 11+):
          Unfreeze ALL layers. Train end-to-end with 10× lower LR.

    Custom FC head: 2048 → 512 → 256 → num_classes (with Dropout).
    """

    def __init__(self, num_classes=10, pretrained=True, dropout=0.5):
        super().__init__()
        self.backbone = models.resnet50(pretrained=pretrained)
        feat_dim = self.backbone.fc.in_features   # 2048

        self.backbone.fc = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(512, 256),      nn.ReLU(inplace=True), nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )
        self._freeze_early()

    def _freeze_early(self):
        """Freeze stem and first two residual stages."""
        for name, param in self.backbone.named_parameters():
            if any(x in name for x in ["conv1", "bn1", "layer1", "layer2"]):
                param.requires_grad = False

    def unfreeze_all(self):
        """Enable gradients for every parameter (Stage 2)."""
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.backbone(x)


_m = ResNet50Genre(10).to(DEVICE)
_d = torch.randn(2, 3, 128, 256).to(DEVICE)
print("ResNet-50 output shape:", _m(_d).shape)
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
total     = sum(p.numel() for p in _m.parameters())
print(f"ResNet-50 trainable: {trainable:,} / {total:,}")
del _m, _d; torch.cuda.empty_cache()

ResNet-50 output shape: torch.Size([2, 10])
ResNet-50 trainable: 19,176,714 / 24,691,018


In [67]:
class ASTGenreModel(nn.Module):
    """
    Audio Spectrogram Transformer for 10-class genre classification.

    The pretrained AST expects 128×1024 input (AudioSet standard).
    Our spectrograms are 128×256 — different number of patches → position
    embedding size mismatch. Fix: interpolate position embeddings to match
    our input size before loading weights, using the HF config overrides.

    If HF download fails we fall back to timm ViT-Small.
    """

    def __init__(self, num_classes=10,
                 hf_model="MIT/ast-finetuned-audioset-10-10-0.4593"):
        super().__init__()
        self.use_hf = False

        if HF_AVAILABLE:
            try:
                from transformers import ASTConfig

                # Load config and override the input size to match our spectrograms.
                # max_length controls time dimension, num_mel_bins controls frequency.
                # HF AST uses 16×16 patches, so:
                #   freq patches = 128 / 16 = 8  (but AST uses overlap: (128-16)/10+1 = 12)
                #   time patches = 256 / 16 = 16  (with overlap: (256-16)/10+1 = 25)
                # Easiest fix: tell the config our actual input dimensions so it
                # builds position embeddings of the right size from scratch.
                config = ASTConfig.from_pretrained(hf_model)
                config.num_mel_bins   = 128   # our spectrogram height
                config.max_length     = 256   # our spectrogram width
                config.num_labels     = num_classes

                # Load with ignore_mismatched_sizes=True so:
                # - Classifier head (527→10) is replaced
                # - Position embeddings (1214→302) are randomly re-initialised
                # - All transformer weights are loaded correctly
                self.model = ASTForAudioClassification.from_pretrained(
                    hf_model,
                    config=config,
                    ignore_mismatched_sizes=True,
                )

                # Freeze first 4 of 12 transformer encoder layers
                for i, layer in enumerate(
                    self.model.audio_spectrogram_transformer.encoder.layer
                ):
                    if i < 4:
                        for p in layer.parameters():
                            p.requires_grad = False

                self.use_hf = True
                print("Loaded HuggingFace AST model ✓")
            except Exception as e:
                print(f"HF AST load failed ({e}) — using ViT-Small fallback")

        if not self.use_hf:
            # timm ViT-Small — same transformer architecture, no size issues
            self.model = timm.create_model(
                "vit_small_patch16_224",
                pretrained=True,
                num_classes=num_classes,
                img_size=(128, 256),
            )
            for blk in self.model.blocks[:4]:
                for p in blk.parameters():
                    p.requires_grad = False
            print("Using timm ViT-Small as AST substitute ✓")

    def forward(self, x):
        if self.use_hf:
            # HF AST internally does unsqueeze(1), so pass (B, H, W) not (B, 1, H, W)
            return self.model(input_values=x[:, 0, :, :]).logits
        return self.model(x)

In [79]:
# ===========================================================================
# Training Engine — with live progress bars
# ===========================================================================

def train_epoch(model, loader, optimizer, criterion, scaler, device, epoch, total_epochs):
    """
    One full training pass with visible per-batch progress bar.
    Shows live loss and accuracy so you can see progress within each epoch.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader,
                desc=f"  Ep {epoch}/{total_epochs} [Train]",
                leave=True,
                dynamic_ncols=True)

    for batch_idx, (specs, labels) in enumerate(pbar):
        specs, labels = specs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            logits = model(specs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = logits.argmax(1)
        correct    += preds.eq(labels).sum().item()
        total      += labels.size(0)
        all_preds  .extend(preds.cpu().numpy())
        all_labels .extend(labels.cpu().numpy())

        pbar.set_postfix({
            "loss": f"{total_loss / (batch_idx + 1):.4f}",
            "acc":  f"{correct / total:.3f}",
        }, refresh=True)

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), correct / total, f1


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, epoch, total_epochs):
    """
    One full validation pass with visible progress bar.
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader,
                desc=f"           [Val  ]",
                leave=True,
                dynamic_ncols=True)

    for batch_idx, (specs, labels) in enumerate(pbar):
        specs, labels = specs.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            logits = model(specs)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        preds       = logits.argmax(1)
        correct    += preds.eq(labels).sum().item()
        total      += labels.size(0)
        all_preds  .extend(preds.cpu().numpy())
        all_labels .extend(labels.cpu().numpy())

        pbar.set_postfix({
            "loss": f"{total_loss / (batch_idx + 1):.4f}",
            "acc":  f"{correct / total:.3f}",
        }, refresh=True)

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), correct / total, f1


def run_training(model, model_key, num_epochs, lr, wd=None):
    """
    Full training loop with:
    - AdamW optimizer + OneCycleLR scheduler
    - AMP mixed precision
    - CrossEntropy with label smoothing
    - Per-epoch WandB logging
    - Best model checkpoint by val macro F1
    - Stage-2 unfreeze for ResNet-50 at epoch 11
    """
    if wd is None:
        wd = CONFIG["weight_decay"]

    print(f"\n{'='*60}")
    print(f"  Training: {model_key.upper()}")
    print(f"  Epochs={num_epochs}  LR={lr}  WD={wd}")
    print(f"{'='*60}")

    wb        = init_wandb_run(model_key, {"lr": lr, "wd": wd, "epochs": num_epochs})
    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=wd,
    )
    scheduler = OneCycleLR(
        optimizer, max_lr=lr,
        epochs=num_epochs, steps_per_epoch=len(train_dl),
        pct_start=0.1, anneal_strategy="cos",
    )
    scaler  = torch.cuda.amp.GradScaler()
    best_f1 = 0.0
    ckpt    = f"/kaggle/working/best_{model_key}.pth"
    history = {k: [] for k in
               ["train_loss", "val_loss", "train_acc", "val_acc", "train_f1", "val_f1"]}

    for epoch in range(1, num_epochs + 1):

        # ResNet-50 Stage 2: unfreeze all layers at epoch 11
        if model_key == "resnet50" and epoch == 11:
            print("  [Stage 2] Unfreezing all ResNet-50 layers…")
            model.unfreeze_all()
            for pg in optimizer.param_groups:
                pg["lr"] = lr * 0.1

        tr_loss, tr_acc, tr_f1 = train_epoch(
            model, train_dl, optimizer, criterion, scaler, DEVICE, epoch, num_epochs)

        scheduler.step()

        vl_loss, vl_acc, vl_f1 = eval_epoch(
            model, val_dl, criterion, DEVICE, epoch, num_epochs)

        # Log to WandB
        wb.log({
            "epoch":          epoch,
            "train/loss":     tr_loss,
            "train/accuracy": tr_acc,
            "train/macro_f1": tr_f1,
            "val/loss":       vl_loss,
            "val/accuracy":   vl_acc,
            "val/macro_f1":   vl_f1,
            "lr":             optimizer.param_groups[0]["lr"],
        })

        for key, val in zip(
            ["train_loss", "val_loss", "train_acc", "val_acc", "train_f1", "val_f1"],
            [tr_loss, vl_loss, tr_acc, vl_acc, tr_f1, vl_f1]
        ):
            history[key].append(val)

        marker = ""
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), ckpt)
            wb.summary["best_val_macro_f1"] = best_f1
            marker = " ← BEST"

        print(
            f"  Ep {epoch:02d}/{num_epochs} | "
            f"Tr Loss {tr_loss:.4f}  F1 {tr_f1:.4f} | "
            f"Val Loss {vl_loss:.4f}  F1 {vl_f1:.4f}{marker}"
        )

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print(f"\n  Best Val Macro F1 [{model_key}]: {best_f1:.4f}")

    art = wandb.Artifact(f"model_{model_key}", type="model")
    art.add_file(ckpt)
    wb.log_artifact(art)
    wb.finish()

    return model, history, best_f1


print("Training engine ready!")

Training engine ready!


In [80]:
print("\n>>> TRAINING MODEL 1: Custom CNN (from scratch) <<<")

cnn_model = ScratchCNN(num_classes=CONFIG["num_classes"], dropout=0.5).to(DEVICE)

cnn_model, cnn_hist, cnn_best = run_training(
    model=cnn_model,
    model_key="scratch_cnn",
    num_epochs=CONFIG["num_epochs_cnn"],
    lr=CONFIG["lr_cnn"],
)

print(f"\n[CNN] Best Val Macro F1 = {cnn_best:.4f}")


>>> TRAINING MODEL 1: Custom CNN (from scratch) <<<

  Training: SCRATCH_CNN
  Epochs=5  LR=0.001  WD=0.0001


<IPython.core.display.Javascript object>

KeyboardInterrupt: 

In [ ]:
print("\n>>> TRAINING MODEL 2: ResNet-50 (Transfer Learning) <<<")

resnet_model = ResNet50Genre(
    num_classes=CONFIG["num_classes"], pretrained=True, dropout=0.5
).to(DEVICE)

resnet_model, resnet_hist, resnet_best = run_training(
    model=resnet_model,
    model_key="resnet50",
    num_epochs=CONFIG["num_epochs_resnet"],
    lr=CONFIG["lr_resnet"],
)

print(f"\n[ResNet-50] Best Val Macro F1 = {resnet_best:.4f}")

In [ ]:
print("\n>>> TRAINING MODEL 3: Audio Spectrogram Transformer (AST) <<<")

ast_model = ASTGenreModel(num_classes=CONFIG["num_classes"]).to(DEVICE)

ast_model, ast_hist, ast_best = run_training(
    model=ast_model,
    model_key="ast",
    num_epochs=CONFIG["num_epochs_ast"],
    lr=CONFIG["lr_ast"],
    wd=1e-2,   # higher WD helps stabilise transformer fine-tuning
)

print(f"\n[AST] Best Val Macro F1 = {ast_best:.4f}")

In [ ]:
# Run num_tta forward passes per sample.
# Pass 1: clean spectrogram.
# Pass 2+: light SpecAugmented versions.
# Average softmax probabilities across all passes → more robust predictions.

@torch.no_grad()
def predict_with_tta(model, loader, device, n_tta=CONFIG["num_tta"]) -> np.ndarray:
    """
    Generate probability predictions with TTA.
    Returns: np.ndarray shape (N_test, num_classes) — averaged probs.
    """
    model.eval()
    all_pass_probs = []

    for tta_i in range(n_tta):
        round_probs = []

        for batch in tqdm(loader, desc=f"TTA {tta_i+1}/{n_tta}", leave=False):
            # TestDataset returns only a tensor (no label)
            specs = batch[0].to(device) if isinstance(batch, (list, tuple)) else batch.to(device)

            if tta_i > 0:
                # Light SpecAugment for augmentation passes
                np_s = specs.cpu().numpy()
                aug_batch = []
                for s in np_s:
                    sa = spec_augment(s[0], freq_param=10, time_param=20, n_masks=1)
                    aug_batch.append(np.stack([sa, sa, sa], axis=0))
                specs = torch.FloatTensor(np.array(aug_batch)).to(device)

            with torch.cuda.amp.autocast():
                logits = model(specs)
            round_probs.append(F.softmax(logits, dim=1).cpu().numpy())

        all_pass_probs.append(np.vstack(round_probs))

    return np.mean(all_pass_probs, axis=0)


print("Generating TTA predictions for all 3 models…")

print("\n[1/3] Custom CNN…")
cnn_probs    = predict_with_tta(cnn_model,    test_dl, DEVICE)

print("\n[2/3] ResNet-50…")
resnet_probs = predict_with_tta(resnet_model, test_dl, DEVICE)

print("\n[3/3] AST / ViT…")
ast_probs    = predict_with_tta(ast_model,    test_dl, DEVICE)

print("\nAll predictions done!")

In [ ]:
# Blend each model's probability vectors weighted by its validation Macro F1.
# Better models naturally get more say in the final prediction.

total_f1 = cnn_best + resnet_best + ast_best + 1e-10
w_cnn    = cnn_best    / total_f1
w_resnet = resnet_best / total_f1
w_ast    = ast_best    / total_f1

print(f"\nEnsemble weights (∝ Val Macro F1):")
print(f"  Custom CNN : {w_cnn:.4f}")
print(f"  ResNet-50  : {w_resnet:.4f}")
print(f"  AST / ViT  : {w_ast:.4f}")

ensemble_probs = w_cnn * cnn_probs + w_resnet * resnet_probs + w_ast * ast_probs
pred_indices   = ensemble_probs.argmax(axis=1)
pred_genres    = label_encoder.inverse_transform(pred_indices)

submission = pd.DataFrame({
    "id":    test_df["id"].values,
    "genre": pred_genres,
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))
print("\nPredicted genre distribution:")
print(submission["genre"].value_counts().to_string())

sub_path = "/kaggle/working/submission.csv"
submission.to_csv(sub_path, index=False)
print(f"\nSubmission saved → {sub_path}")

In [ ]:
def plot_history(history, title, save_path=None):
    """Plot loss and macro-F1 training curves for one model."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], "b-", label="Train")
    axes[0].plot(epochs, history["val_loss"],   "r-", label="Val")
    axes[0].set(title="Loss", xlabel="Epoch", ylabel="CE Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history["train_f1"], "b-", label="Train")
    axes[1].plot(epochs, history["val_f1"],   "r-", label="Val")
    axes[1].set(title="Macro F1", xlabel="Epoch", ylabel="F1")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


plot_history(cnn_hist,    "Custom CNN — Training History",  "/kaggle/working/cnn_history.png")
plot_history(resnet_hist, "ResNet-50  — Training History",  "/kaggle/working/resnet_history.png")
plot_history(ast_hist,    "AST / ViT  — Training History",  "/kaggle/working/ast_history.png")


def plot_cm(model, loader, device, le, model_name, save_path=None):
    """Confusion matrix + classification report on validation set."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for specs, labels in loader:
            specs = specs.to(device)
            with torch.cuda.amp.autocast():
                logits = model(specs)
            all_preds .extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set(xlabel="Predicted", ylabel="True", title=f"CM — {model_name}")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nClassification Report — {model_name}:")
    print(classification_report(all_labels, all_preds,
                                target_names=le.classes_, zero_division=0))


plot_cm(cnn_model,    val_dl, DEVICE, label_encoder, "Custom CNN",  "/kaggle/working/cm_cnn.png")
plot_cm(resnet_model, val_dl, DEVICE, label_encoder, "ResNet-50",   "/kaggle/working/cm_resnet.png")
plot_cm(ast_model,    val_dl, DEVICE, label_encoder, "AST / ViT",   "/kaggle/working/cm_ast.png")

In [ ]:
# Upload all plots + submission + final metrics to a dedicated WandB run.

print("\nLogging ensemble summary to WandB…")

summary_run = wandb.init(
    project=CONFIG["wandb_project"],
    name="ensemble_summary",
    config={
        "cnn_best_f1":    cnn_best,
        "resnet_best_f1": resnet_best,
        "ast_best_f1":    ast_best,
        "w_cnn":          float(w_cnn),
        "w_resnet":       float(w_resnet),
        "w_ast":          float(w_ast),
        "n_test":         len(submission),
    },
    reinit=True,
)

summary_run.log({
    "cnn/val_macro_f1":    cnn_best,
    "resnet/val_macro_f1": resnet_best,
    "ast/val_macro_f1":    ast_best,
})

for tag, path in [
    ("plot/cnn_history",    "/kaggle/working/cnn_history.png"),
    ("plot/resnet_history", "/kaggle/working/resnet_history.png"),
    ("plot/ast_history",    "/kaggle/working/ast_history.png"),
    ("plot/cm_cnn",         "/kaggle/working/cm_cnn.png"),
    ("plot/cm_resnet",      "/kaggle/working/cm_resnet.png"),
    ("plot/cm_ast",         "/kaggle/working/cm_ast.png"),
]:
    summary_run.log({tag: wandb.Image(path)})

art = wandb.Artifact("submission", type="submission")
art.add_file(sub_path)
summary_run.log_artifact(art)
summary_run.finish()

In [ ]:
print("\n" + "=" * 62)
print("  FINAL RESULTS SUMMARY")
print("=" * 62)
print(f"  {'Model':<24} {'Val Macro F1':>14}  {'Weight':>10}")
print(f"  {'-'*52}")
print(f"  {'Custom CNN (Scratch)':<24} {cnn_best:>14.4f}  {w_cnn:>10.4f}")
print(f"  {'ResNet-50':<24} {resnet_best:>14.4f}  {w_resnet:>10.4f}")
print(f"  {'AST / ViT':<24} {ast_best:>14.4f}  {w_ast:>10.4f}")
print(f"  {'─'*52}")
print(f"  Submission : {sub_path}")
print(f"  Test count : {len(submission)}")
print("=" * 62)
print("\nDone! Upload submission.csv to Kaggle to get your leaderboard score.")